In [ ]:
print("hello world")

hello world


#Lab4Q1

In [ ]:
import pandas as pd

rollno = 1024160069
L = []

for i in range(2):
  L.append(rollno%10)
  rollno = rollno//10

L.reverse()
print(L)

fixed_entries = [ {"question": "what is the annual fee",
                   "answer": "The annual fee is Rs 500.",
                   "keywords": "fee cost price charge",
                   "category": "billing"},
                    {"question": "how to reset password",
                     "answer": "Go to Settings > Reset Password.",
                     "keywords": "password reset login",
                     "category": "account"},
                      {"question": "what are your working hours",
                       "answer": "We are open 9 AM to 5 PM.",
                       "keywords": "hours timing open time",
                       "category": "general"},
                        {"question": "how can i pay the fee",
                         "answer": "You can pay via UPI, card, or net banking.",
                         "keywords": "pay payment upi fee",
                         "category": "billing"}, ]

personalised_entries = []

categories = ["billing","account","general"]

for j in L:
  category = categories[j%3]

  if category == "billing":
    entry = {"question":"What is  billing?",
             "answer":"I don't know",
             "keywords":"Billing is something",
             "category":"billing"}
  if category == "account":
    entry = {"question":"What is  account?",
             "answer":"I don't know",
             "keywords":"account is something",
             "category":"account"}
  if category == "general":
    entry = {"question":"What is  general?",
             "answer":"I don't know",
             "keywords":"general is something",
             "category":"general"}
  personalised_entries.append(entry)


all_entries = fixed_entries+personalised_entries

df = pd.DataFrame(all_entries)

df

[6, 9]


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,What is billing?,I don't know,Billing is something,billing
5,What is billing?,I don't know,Billing is something,billing


In [ ]:
def score_hypothesis(query,df):
  query_words = set(query.lower().strip())
  print(query_words)

  results = []

  for _,row in df.iterrows():

    keywords = set(row["keywords"].lower().strip())

    matched_words = query_words.intersection(keywords)

    if len(query_words)>0:
      confidence = len(matched_words)/len(query_words)
    else:
      confidence = 0

    if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(matched_words),
                "confidence": confidence})


  results.sort(key=lambda x: x["confidence"], reverse=True)

  return pd.DataFrame(results)


query = "how can I pay my fee"

result = score_hypothesis(query, df)

result

{'h', 'e', 'a', 'y', 'f', 'i', ' ', 'n', 'o', 'm', 'p', 'w', 'c'}


,question,answer,category,matched_keywords,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,"h, e, a, f, i, , o, p, c",0.692308
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,"e, a, y, f, i, , n, m, p",0.692308
2,how to reset password,Go to Settings > Reset Password.,account,"e, a, i, , n, o, p, w",0.615385
3,what are your working hours,We are open 9 AM to 5 PM.,general,"h, e, i, , n, o, m, p",0.615385
4,What is billing?,I don't know,billing,"h, e, i, , n, o, m",0.538462
5,What is billing?,I don't know,billing,"h, e, i, , n, o, m",0.538462


#Lab4Q3

In [ ]:
def same_category(category_name,df):

  result = []

  for _,row in df.iterrows():
    if row["category"] == category_name:
      result.append({
          "question":row["question"],
          "answer":row["answer"],
          "keywords":row["keywords"],
          "category":row["category"]
      })


  return pd.DataFrame(result)


answer = same_category("billing",df)
answer

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
2,What is billing?,I don't know,Billing is something,billing
3,What is billing?,I don't know,Billing is something,billing


#Lab4Q4

In [ ]:
df_new = df.copy()

a = input("Enter something for keyword:")

df_new.iloc[4,2] = a

df_new.to_csv("1024160069_faq_data.csv",index=False)


Enter something for keyword:gbruhhhhh


#Lab4Q5

In [ ]:
category_counts = df.groupby("category").size()

print(category_counts)

category
account    1
billing    4
general    1
dtype: int64


#Lab4Q6

In [ ]:
def score_query_with_tie(query, df):

    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        keyword_matches = query_words.intersection(keyword_words)
        question_matches = query_words.intersection(question_words)

        score = len(keyword_matches) * 2 + len(question_matches)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching FAQ found.")
        return


    highest_score = max(result["score"] for result in results)
    top_results = [
        result for result in results
        if result["score"] == highest_score
    ]

    print("\nHighest Confidence Score:", highest_score)

    if len(top_results) > 1:
        print("\nTIE DETECTED! Multiple matching entries:")

        for result in top_results:
            print("\nQuestion:", result["question"])
            print("Answer:", result["answer"])
            print("Category:", result["category"])
            print("Score:", result["score"])

    else:
        print("\nBest Matching Entry:")

        result = top_results[0]

        print("Question:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Score:", result["score"])



print("\n--- Query producing a TIE ---")
score_query_with_tie("fee", df)

print("\n--- Query NOT producing a TIE ---")
score_query_with_tie("password", df)
